# Analisis HR Employee Attrition (People Analytics)

**Dataset:** IBM HR Analytics Employee Attrition & Performance (dataset publik, 1.470 karyawan, 35 kolom).
**Tujuan:** memahami segmen karyawan mana yang paling sering resign agar perusahaan bisa menyusun strategi retensi yang tepat sasaran.

> **Definisi metrik:** karyawan disebut *resign* jika kolom `Attrition` bernilai `Yes`.
> *Attrition rate* = jumlah resign / total karyawan pada segmen tersebut.

Alur kerja: data mentah dimuat ke SQLite (`data/hr_attrition.db`) lewat `scripts/load_to_sqlite.py`,
lalu setiap pertanyaan bisnis dijawab dengan **query SQL asli** yang hasilnya ditampilkan sebagai tabel di bawah ini.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display, Image

# Koneksi ke database hasil load_to_sqlite.py
DB_PATH = Path("data/hr_attrition.db")
con = sqlite3.connect(DB_PATH)

def sql(query):
    """Jalankan query SQL terhadap tabel `employees`, tampilkan sebagai tabel."""
    return pd.read_sql_query(query, con)


## 1. Attrition Rate Keseluruhan

Berapa besar tingkat resign di perusahaan secara keseluruhan? Ini menjadi angka pembanding untuk semua segmen berikutnya.

In [ ]:
sql("""
SELECT
    COUNT(*)                                        AS total_karyawan,
    SUM(Attrition = 'Yes')                          AS resign,
    ROUND(100.0 * SUM(Attrition = 'Yes') / COUNT(*), 1) AS rate_persen
FROM employees;
""")

In [ ]:
display(Image("charts/01_attrition_overall.png"))

**Insight:** dari 1.470 karyawan, **237 orang resign** sehingga attrition rate keseluruhan adalah **16,1%**. Setiap segmen dengan rate di atas angka ini patut mendapat perhatian khusus.

## 2. Attrition per Departemen dan Job Role

Departemen dan role mana yang paling banyak kehilangan karyawan?

In [ ]:
sql("""
SELECT
    Department,
    COUNT(*)                       AS karyawan,
    SUM(Attrition = 'Yes')         AS resign,
    ROUND(100.0 * SUM(Attrition = 'Yes') / COUNT(*), 1) AS rate_persen
FROM employees
GROUP BY Department
ORDER BY rate_persen DESC;
""")

In [ ]:
display(Image("charts/02_attrition_department.png"))

In [ ]:
sql("""
SELECT
    JobRole,
    COUNT(*)                       AS karyawan,
    SUM(Attrition = 'Yes')         AS resign,
    ROUND(100.0 * SUM(Attrition = 'Yes') / COUNT(*), 1) AS rate_persen
FROM employees
GROUP BY JobRole
ORDER BY rate_persen DESC;
""")

In [ ]:
display(Image("charts/03_attrition_jobrole.png"))

**Insight:** departemen **Sales** tertinggi (20,6%, 92 dari 446 karyawan), disusul Human Resources (19,0%) dan R&D (13,8%).
Di level role, **Sales Representative** jauh di atas yang lain: **39,8%** (33 dari 83 orang).
Laboratory Technician (23,9%, 62 dari 259) dan Human Resources (23,1%, 12 dari 52) juga di atas rata-rata,
sedangkan posisi direktur dan manajer sangat stabil (2,5-6,9%).

## 3. OverTime (Lembur) vs Attrition

Apakah karyawan yang lembur lebih rentan resign? Kolom `OverTime` bernilai `Yes`/`No`.

In [ ]:
sql("""
SELECT
    OverTime,
    COUNT(*)                       AS karyawan,
    SUM(Attrition = 'Yes')         AS resign,
    ROUND(100.0 * SUM(Attrition = 'Yes') / COUNT(*), 1) AS rate_persen
FROM employees
GROUP BY OverTime;
""")

In [ ]:
display(Image("charts/04_attrition_overtime.png"))

**Insight:** karyawan yang lembur resign **30,5%** (127 dari 416), hampir **3 kali lipat** dibanding yang tidak lembur (10,4%, 110 dari 1.054).
Ini adalah salah satu sinyal risiko terkuat dalam dataset ini.

## 4. Band Gaji Bulanan (MonthlyIncome) vs Attrition

Gaji dikelompokkan ke band `< 3.000`, `3.000-6.000`, `6.000-10.000`, dan `> 10.000` (USD).

In [ ]:
sql("""
SELECT
    CASE
        WHEN MonthlyIncome < 3000  THEN '< 3.000'
        WHEN MonthlyIncome < 6000  THEN '3.000-6.000'
        WHEN MonthlyIncome < 10000 THEN '6.000-10.000'
        ELSE '> 10.000'
    END AS band_gaji,
    COUNT(*)                       AS karyawan,
    SUM(Attrition = 'Yes')         AS resign,
    ROUND(100.0 * SUM(Attrition = 'Yes') / COUNT(*), 1) AS rate_persen
FROM employees
GROUP BY band_gaji
ORDER BY MIN(MonthlyIncome);
""")

In [ ]:
display(Image("charts/05_attrition_income_band.png"))

**Insight:** pola menurun jelas — karyawan bergaji **< 3.000 resign 28,6%** (113 dari 395),
sementara band > 10.000 hanya 8,9% (25 dari 281). Gaji rendah berkorelasi kuat dengan niat keluar.

## 5. Kelompok Umur vs Attrition

Umur dikelompokkan: `< 30`, `30-39`, `40-49`, `50+` tahun.

In [ ]:
sql("""
SELECT
    CASE
        WHEN Age < 30 THEN '< 30'
        WHEN Age < 40 THEN '30-39'
        WHEN Age < 50 THEN '40-49'
        ELSE '50+'
    END AS kelompok_umur,
    COUNT(*)                       AS karyawan,
    SUM(Attrition = 'Yes')         AS resign,
    ROUND(100.0 * SUM(Attrition = 'Yes') / COUNT(*), 1) AS rate_persen
FROM employees
GROUP BY kelompok_umur
ORDER BY MIN(Age);
""")

In [ ]:
display(Image("charts/06_attrition_age_group.png"))

**Insight:** karyawan **di bawah 30 tahun** paling rentan: **27,9%** (91 dari 326).
Kelompok 40-49 paling stabil (9,7%). Karyawan muda tampaknya lebih mudah pindah kerja.

## 6. Jarak Rumah ke Kantor vs Attrition

`DistanceFromHome` (km) dikelompokkan: `1-4`, `5-9`, `10-14`, `15+`.

In [ ]:
sql("""
SELECT
    CASE
        WHEN DistanceFromHome < 5  THEN '1-4'
        WHEN DistanceFromHome < 10 THEN '5-9'
        WHEN DistanceFromHome < 15 THEN '10-14'
        ELSE '15+'
    END AS kelompok_jarak,
    COUNT(*)                       AS karyawan,
    SUM(Attrition = 'Yes')         AS resign,
    ROUND(100.0 * SUM(Attrition = 'Yes') / COUNT(*), 1) AS rate_persen
FROM employees
GROUP BY kelompok_jarak
ORDER BY MIN(DistanceFromHome);
""")

In [ ]:
display(Image("charts/07_attrition_distance.png"))

**Insight:** semakin jauh jarak tempuh, semakin tinggi resign — naik bertahap dari 13,6% (jarak 1-4 km)
ke **20,6%** (jarak 15+ km, 73 dari 355). Jarak jauh memperberat beban harian karyawan.

## 7. Kepuasan Kerja, Lingkungan, dan Work-Life Balance vs Attrition

Tiga kolom kepuasan dalam dataset ini tersimpan sebagai label (bukan angka 1-4):
`JobSatisfaction` dan `EnvironmentSatisfaction` memakai label *Low < Medium < High < Very High*,
sedangkan `WorkLifeBalance` memakai label *Bad < Good < Better < Best*.

In [ ]:
sql("""
SELECT
    JobSatisfaction,
    COUNT(*)                       AS karyawan,
    SUM(Attrition = 'Yes')         AS resign,
    ROUND(100.0 * SUM(Attrition = 'Yes') / COUNT(*), 1) AS rate_persen
FROM employees
GROUP BY JobSatisfaction;
""")

In [ ]:
display(Image("charts/08_attrition_job_satisfaction.png"))

In [ ]:
sql("""
SELECT
    EnvironmentSatisfaction,
    COUNT(*)                       AS karyawan,
    SUM(Attrition = 'Yes')         AS resign,
    ROUND(100.0 * SUM(Attrition = 'Yes') / COUNT(*), 1) AS rate_persen
FROM employees
GROUP BY EnvironmentSatisfaction;
""")

In [ ]:
display(Image("charts/09_attrition_env_satisfaction.png"))

In [ ]:
sql("""
SELECT
    WorkLifeBalance,
    COUNT(*)                       AS karyawan,
    SUM(Attrition = 'Yes')         AS resign,
    ROUND(100.0 * SUM(Attrition = 'Yes') / COUNT(*), 1) AS rate_persen
FROM employees
GROUP BY WorkLifeBalance;
""")

In [ ]:
display(Image("charts/10_attrition_worklife_balance.png"))

**Insight:** kepuasan rendah selalu berpasangan dengan resign tinggi —
`JobSatisfaction=Low` 22,8%, `EnvironmentSatisfaction=Low` 25,4% (tertinggi kedua untuk faktor tunggal),
dan `WorkLifeBalance=Bad` **31,3%** (25 dari 80 karyawan). Karyawan yang merasa hidup kerjanya tidak seimbang adalah segmen paling rawan.

## 8. Masa Kerja, Lama di Posisi, dan Jarak Promosi vs Attrition

Tiga ukuran tenure: `YearsAtCompany` (masa kerja di perusahaan), `YearsInCurrentRole` (lama di posisi sekarang),
dan `YearsSinceLastPromotion` (jarak sejak promosi terakhir).

In [ ]:
sql("""
SELECT
    CASE
        WHEN YearsAtCompany < 3  THEN '0-2'
        WHEN YearsAtCompany < 6  THEN '3-5'
        WHEN YearsAtCompany < 11 THEN '6-10'
        ELSE '11+'
    END AS masa_kerja,
    COUNT(*)                       AS karyawan,
    SUM(Attrition = 'Yes')         AS resign,
    ROUND(100.0 * SUM(Attrition = 'Yes') / COUNT(*), 1) AS rate_persen
FROM employees
GROUP BY masa_kerja
ORDER BY MIN(YearsAtCompany);
""")

In [ ]:
sql("""
SELECT
    CASE
        WHEN YearsInCurrentRole < 3 THEN '0-2'
        WHEN YearsInCurrentRole < 6 THEN '3-5'
        ELSE '6+'
    END AS lama_di_posisi,
    COUNT(*)                       AS karyawan,
    SUM(Attrition = 'Yes')         AS resign,
    ROUND(100.0 * SUM(Attrition = 'Yes') / COUNT(*), 1) AS rate_persen
FROM employees
GROUP BY lama_di_posisi
ORDER BY MIN(YearsInCurrentRole);
""")

In [ ]:
sql("""
SELECT
    CASE
        WHEN YearsSinceLastPromotion < 3 THEN '0-2'
        WHEN YearsSinceLastPromotion < 6 THEN '3-5'
        ELSE '6+'
    END AS sejak_promosi,
    COUNT(*)                       AS karyawan,
    SUM(Attrition = 'Yes')         AS resign,
    ROUND(100.0 * SUM(Attrition = 'Yes') / COUNT(*), 1) AS rate_persen
FROM employees
GROUP BY sejak_promosi
ORDER BY MIN(YearsSinceLastPromotion);
""")

In [ ]:
display(Image("charts/11_attrition_tenure.png"))

**Insight:** tahun-tahun awal adalah masa paling rawan — masa kerja **0-2 tahun: 29,8%** (102 dari 342),
menurun bertahap hingga 8,1% untuk masa kerja 11+ tahun. Pola sama terlihat pada lama di posisi
(0-2 tahun: 22,6%) dan jarak promosi (0-2 tahun: 17,0%; menariknya yang 6+ tahun sejak promosi naik lagi ke 16,3%,
sinyal stagnasi karier pada karyawan senior).

## 9. Gender, Status Perkawinan, dan Perjalanan Dinas vs Attrition

In [ ]:
sql("""
SELECT
    Gender,
    COUNT(*)                       AS karyawan,
    SUM(Attrition = 'Yes')         AS resign,
    ROUND(100.0 * SUM(Attrition = 'Yes') / COUNT(*), 1) AS rate_persen
FROM employees
GROUP BY Gender;
""")

In [ ]:
sql("""
SELECT
    MaritalStatus,
    COUNT(*)                       AS karyawan,
    SUM(Attrition = 'Yes')         AS resign,
    ROUND(100.0 * SUM(Attrition = 'Yes') / COUNT(*), 1) AS rate_persen
FROM employees
GROUP BY MaritalStatus
ORDER BY rate_persen DESC;
""")

In [ ]:
sql("""
SELECT
    BusinessTravel,
    COUNT(*)                       AS karyawan,
    SUM(Attrition = 'Yes')         AS resign,
    ROUND(100.0 * SUM(Attrition = 'Yes') / COUNT(*), 1) AS rate_persen
FROM employees
GROUP BY BusinessTravel
ORDER BY rate_persen DESC;
""")

In [ ]:
display(Image("charts/12_attrition_demografi_travel.png"))

**Insight:** gender hampir tidak berpengaruh (laki-laki 17,0% vs perempuan 14,8%).
Karyawan **single resign 25,5%** (120 dari 470), dua kali lipat married (12,5%) dan divorced (10,1%).
Pola perjalanan dinas juga tajam: yang **sering dinas (Travel_Frequently) 24,9%** (69 dari 277),
dibanding jarang dinas 15,0% dan tidak dinas hanya 8,0%.

## 10. Temuan Kunci

| # | Temuan | Angka |
|---|--------|-------|
| 1 | Attrition keseluruhan | **16,1%** (237 dari 1.470) |
| 2 | Role paling rawan: Sales Representative | **39,8%** (33 dari 83) |
| 3 | Karyawan lembur resign hampir 3x lipat | **30,5%** vs 10,4% |
| 4 | Karyawan baru (masa kerja 0-2 tahun) | **29,8%** (102 dari 342) |
| 5 | Gaji < 3.000 | **28,6%** (113 dari 395) |
| 6 | Umur < 30 tahun | **27,9%** (91 dari 326) |
| 7 | Single | **25,5%** (120 dari 470) |
| 8 | Sering perjalanan dinas | **24,9%** (69 dari 277) |
| 9 | Kepuasan lingkungan rendah | **25,4%** (72 dari 284) |
| 10 | Work-life balance buruk | **31,3%** (25 dari 80) |
| 11 | Jarak rumah 15+ km | **20,6%** (73 dari 355) |

Segmen paling berisiko adalah **karyawan muda, baru masuk, bergaji rendah, sering lembur/dinas, dan berstatus single**
— profil yang saling tumpang tindih (mis. Sales Representative muda yang lembur).

## 11. Rekomendasi Retensi

1. **Program onboarding 2 tahun pertama yang terstruktur.** Masa kerja 0-2 tahun resign 29,8% — tetapkan buddy/mentor,
   tinjau beban kerja 90 hari pertama, dan jadwalkan *check-in* karier berkala agar karyawan baru cepat merasa terhubung.
2. **Kelola lembur secara ketat, terutama di Sales dan Lab.** Lembur menaikkan resign hingga 30,5%.
   Batasi jam lembur per bulan, beri kompensasi cuti pengganti, dan pantau tim dengan lembur kronis sebagai *early warning*.
3. **Perbaiki paket kompensasi dan jenjang karier role berisiko tinggi.** Sales Representative (39,8%) dan
   Laboratory Technician (23,9%) bergaji relatif rendah — tinjau ulang struktur gaji, bonus berbasis kinerja,
   dan jalur promosi yang jelas; karyawan 6+ tahun tanpa promosi pun kembali berisiko (16,3%).
4. **Fleksibilitas untuk karyawan yang jauh dan sering dinas.** Jarak 15+ km (20,6%) dan sering dinas (24,9%)
   menambah beban harian — pertimbangkan opsi hybrid/WFH parsial, tunjangan transportasi, dan rotasi jadwal dinas
   agar tidak menumpuk pada orang yang sama.
5. **Intervensi dini berbasis survei kepuasan.** Kepuasan lingkungan *Low* (25,4%) dan work-life balance *Bad* (31,3%)
   adalah sinyal yang bisa dideteksi sebelum resign — tindak lanjuti hasil survei per tim dengan rencana perbaikan
   konkret, bukan sekadar laporan tahunan.

In [ ]:
con.close()
print("Analisis selesai.")